# Splink on Snowflake: Lift-and-Shift with Java UDF JARs

This notebook demonstrates running [Splink v4](https://github.com/moj-analytical-services/splink)
on Snowflake via Snowpark Connect for Spark (SCOS) using Splink's built-in Java UDF JARs
for `jaro_winkler` and `damerau_levenshtein` string similarity.

**What you'll learn:**
- How to initialize a Spark session connected to Snowflake
- How to provision Java UDF JARs (similarity functions + Scala runtime)
- How to configure and run the full Splink pipeline (train, predict, cluster)

**Prerequisites:**
- Run `00_setup.sql` first to create the relevant resources (Database, Compute Pool, Warehouse, etc.)
- Upload the accompanying `pyproject.toml` into the same directory as this notebook

**Comparisons used:**
- `ForenameSurnameComparison` (first_name + surname) — multi-column, columns-reversed, JW (JAR)
- `NameComparison` (full_name) — JW at 3 thresholds (JAR)
- `DateOfBirthComparison` (dob) — DamerauLevenshtein (JAR) + date arithmetic (native)
- `PostcodeComparison` (postcode_fake) — hierarchical regex matching (native, no JAR)
- `LevenshteinAtThresholds` (occupation) — built-in Spark function (no JAR)
- `ExactMatch` (gender) — equality (native)
- `ExactMatch` (birth_place) with TF adjustments — equality + frequency weighting (native)

**Dataset:** `splink_datasets.historical_50k` (50,000 rows)  
**Warehouse:** `SPLINK_WH` (MEDIUM)

In [ ]:
%pip install uv

In [ ]:
!uv pip install --system -r pyproject.toml

In [ ]:
import os, glob
import pyspark
from snowflake.snowpark_connect import init_spark_session
from snowflake.snowpark_connect.snowflake_session import SnowflakeSession
from splink import Linker, SparkAPI, SettingsCreator, block_on, splink_datasets
import splink.comparison_library as cl
from splink.backends.spark import similarity_jar_location

## Connect to Snowflake

Initialize a Spark session backed by Snowflake via Snowpark Connect and set the relevant Spark and Snowpark Connect configurations, followed by your Snowflake context.

In [ ]:
conf = pyspark.SparkConf()
conf.set("spark.sql.session.timeZone", "UTC")
conf.set("spark.sql.ansi.enabled", "false")
conf.set("snowpark.connect.cte.optimization_enabled", "true")
conf.set("snowpark.connect.integralTypesEmulation", "enabled")

spark = init_spark_session(app_name="splink", conf=conf)

In [ ]:
WAREHOUSE = "SPLINK_WH"
DATABASE = "SPLINK_DEMO"
SCHEMA = "PUBLIC"
ROLE = "SPLINK_ROLE"
NUM_PARTITIONS = 8

session = SnowflakeSession(spark)
session.sql(f"USE ROLE {ROLE}").collect()
session.sql(f"USE WAREHOUSE {WAREHOUSE}").collect()
session.sql(f"USE DATABASE {DATABASE}").collect()
session.sql(f"USE SCHEMA {SCHEMA}").collect()

print(f"Session ready. num_partitions_on_repartition={NUM_PARTITIONS}")
session.sql("SELECT CURRENT_WAREHOUSE() AS wh, CURRENT_DATABASE() AS db, CURRENT_SCHEMA() AS sch").show()

## Provision Splink's Java UDF JARs

Two JARs are required: the similarity JAR (jaro_winkler, damerau_levenshtein, etc.) AND the
Scala 2.12 runtime (the SCOS sandbox doesn't bundle it — omitting causes `NoClassDefFoundError`).

**Note:** To set JARs, make sure to use the `spark.addArtifact()` method 

In [ ]:
similarity_jar = similarity_jar_location()
scala_candidates = glob.glob(
    os.path.join(os.path.dirname(pyspark.__file__), "jars", "scala-library-2.12*.jar")
)
assert scala_candidates, "scala-library-2.12*.jar not found under pyspark/jars"
scala_jar = scala_candidates[0]

spark.addArtifact(similarity_jar)
spark.addArtifact(scala_jar)
print(f"Uploaded: {os.path.basename(similarity_jar)}, {os.path.basename(scala_jar)}")

## Load Data

Splink ships with a built-in synthetic dataset (`historical_50k`) designed for
record linkage testing. We load it into a Spark DataFrame backed by Snowflake.

In [ ]:
pdf = splink_datasets.historical_50k
df = spark.createDataFrame(pdf)
row_count = df.count()
print(f"Loaded historical_50k: {row_count:,} rows")
df.printSchema()

## Configure the Linker

Define comparison rules (string similarity, date arithmetic, exact match) and blocking
rules that control which record pairs are evaluated. The `register_udfs_automatically=True`
setting tells Splink to auto-register its Java UDFs using the uploaded JARs.

In [ ]:
db_api = SparkAPI(
    spark_session=spark,
    break_lineage_method="persist",
    register_udfs_automatically=True,
    num_partitions_on_repartition=NUM_PARTITIONS,
)

settings = SettingsCreator(
    link_type="dedupe_only",
    comparisons=[
        cl.ForenameSurnameComparison("first_name", "surname"),
        cl.NameComparison("full_name"),
        cl.DateOfBirthComparison("dob", input_is_string=True),
        cl.PostcodeComparison("postcode_fake"),
        cl.LevenshteinAtThresholds("occupation", [2, 4]),
        cl.ExactMatch("gender"),
        cl.ExactMatch("birth_place").configure(term_frequency_adjustments=True),
    ],
    blocking_rules_to_generate_predictions=[
        block_on("first_name", "dob"),
        block_on("surname", "birth_place"),
        block_on("postcode_fake"),
    ],
    retain_intermediate_calculation_columns=True,
    em_convergence=0.001,
    max_iterations=30,
)

linker = Linker(df, settings, db_api=db_api)
print("Linker constructed. Java UDFs auto-registered.")

## Train the Model

Splink's training consists of four steps:
1. Estimate the probability that two random records match
2. Estimate u-values (agreement rates among non-matches) via random sampling
3. EM pass blocking on (first_name, surname)
4. EM pass blocking on (dob)

In [ ]:
deterministic_rules = [
    block_on("first_name", "surname", "dob"),
    block_on("surname", "birth_place"),
]

linker.training.estimate_probability_two_random_records_match(deterministic_rules, recall=0.7)
linker.training.estimate_u_using_random_sampling(max_pairs=1e8, seed=42)
linker.training.estimate_parameters_using_expectation_maximisation(block_on("first_name", "surname"))
linker.training.estimate_parameters_using_expectation_maximisation(block_on("dob"))
print("Training complete.")

## Predict and Cluster

Run inference to find record pairs with match probability >= 0.85, then cluster
those pairs into entity groups at a stricter 0.95 threshold.

In [ ]:
df_predict = linker.inference.predict(threshold_match_probability=0.85)
n_pairs = df_predict.as_spark_dataframe().count()
print(f"Predicted pairs (p >= 0.85): {n_pairs:,}")

df_clusters = linker.clustering.cluster_pairwise_predictions_at_threshold(
    df_predict, threshold_match_probability=0.95
)
clusters_sdf = df_clusters.as_spark_dataframe()
n_clusters = clusters_sdf.select("cluster_id").distinct().count()
print(f"Distinct clusters (p >= 0.95): {n_clusters:,}")